# Workshop: Delta DML & Time Travel

> *"Merge new customer data, handle accidental deletes, recover using Time Travel, and see how VACUUM affects it."*

## Setup

In [0]:
%run ../../setup/00_setup

In [0]:
# Ensure bronze.customers table exists (idempotent)
customers_path = f"{DATASET_PATH}/customers/customers.csv"
df_base = spark.read.format("csv").option("header", True).option("inferSchema", True).load(customers_path)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.customers")
df_base.write.mode("overwrite").saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.customers")
print(f"Base table ready: {spark.table(f'{CATALOG}.{BRONZE_SCHEMA}.customers').count()} rows")

## Task 1: Examine the Update File

Load `customers_new.csv` and compare counts. How many customers overlap with base table?

**Guidance — Task 01**

Load the customer update file into a DataFrame to inspect what changes are coming in.

**What you need to do**
Use `spark.read` to load `customers_new.csv` into a variable called `df_updates`. The file path is defined for you as `update_path`.

**Hint:** CSV format, `header=True`, `inferSchema=True`, `.load(path)` — same pattern as the preparation cell above.

In [0]:
# TODO: Load customers_new.csv into df_updates
# The path is already defined as: update_path = f"{DATASET_PATH}/customers/customers_new.csv"
update_path = f"{DATASET_PATH}/customers/customers_new.csv"
df_updates = (
    spark.read.format("csv").option("header", True).option("inferSchema", True).load(update_path)
)


print(f"Existing customers: {spark.table(f'{CATALOG}.{BRONZE_SCHEMA}.customers').count()}")
print(f"Updates file: {df_updates.count()} rows")
display(df_updates.limit(5))

In [0]:
# -- Validation --
assert df_updates.count() > 0, "Updates file is empty"
print(f"Task 1 OK: {df_updates.count()} update records loaded")

## Task 2: MERGE INTO (Upsert)

Register `df_updates` as temp view `v_updates`, then use SQL MERGE to:
- **UPDATE** existing customers (match on `customer_id`)
- **INSERT** new customers

**Guidance — Task 02**

`MERGE INTO` (upsert) applies updates and inserts in a single atomic operation — the standard way to sync Delta tables.

**MERGE syntax**
```sql
MERGE INTO <target> AS t
USING <source> AS s
ON t.key = s.key
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
```

`UPDATE SET *` updates all target columns from the source. `INSERT *` inserts the full source row. The source temp view `v_updates` is already registered below.

**Hint:** Use the fully qualified target table name: `{CATALOG}.{BRONZE_SCHEMA}.customers`

In [0]:
# Register updates as temp view
df_updates.createOrReplaceTempView("v_updates")

In [0]:
# TODO: Write a MERGE INTO statement to upsert v_updates into the customers table
# - Match on: customer_id
# - When matched: UPDATE SET *
# - When not matched: INSERT *

spark.sql(f"""
    MERGE INTO {CATALOG}.{BRONZE_SCHEMA}.customers AS t
    USING v_updates AS s
    ON t.customer_id = s.customer_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

new_count = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers").count()
print(f"Customers after MERGE: {new_count}")

In [0]:
# -- Validation --
base_count = df_base.count()
assert new_count >= base_count, f"Expected at least {base_count} rows after MERGE, got {new_count}"
print(f"Task 2 OK: MERGE completed. {new_count} total customers (was {base_count})")

## Task 3: UPDATE Records

Update the `state` column for all customers where `city = 'Austin'`. Set state to `'TX'`.

**Guidance — Task 03**

Delta Lake supports `UPDATE` directly — unlike Parquet files, you can modify rows in place.

**UPDATE syntax**
```sql
UPDATE catalog.schema.table
SET column = 'value'
WHERE condition = 'value'
```

Set `state = 'TX'` for all rows where `city = 'Austin'`.

**Hint:** String literals in SQL must be quoted: `'TX'`, `'Austin'`

In [0]:
# TODO: Update the state column to 'TX' for all customers in Austin

spark.sql(f"""
    UPDATE {CATALOG}.{BRONZE_SCHEMA}.customers
    SET state = 'TX'
    WHERE city = 'Austin'
""")

# Verify
display(spark.sql(f"SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.customers WHERE city = 'Austin'"))

## Task 4: Accidental DELETE

Simulate an accident -- delete all customers where country is not null.

**WARNING:** This is intentional! We will recover the data using Time Travel.

In [0]:
# Record row count BEFORE the accident
count_before = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers").count()
print(f"Rows BEFORE delete: {count_before}")

# "Accident" - delete a large chunk
spark.sql(f"""
    DELETE FROM {CATALOG}.{BRONZE_SCHEMA}.customers
    WHERE country IS NOT NULL
""")

count_after = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers").count()
print(f"Rows AFTER delete: {count_after} (lost {count_before - count_after} rows!)")

## Task 5: DESCRIBE HISTORY

Check the table history to see all operations performed.

**Guidance — Task 05**

Every write operation on a Delta table creates a new entry in the transaction log with a version number.

**DESCRIBE HISTORY syntax**
```sql
DESCRIBE HISTORY catalog.schema.table
```

Returns one row per version with: `version` (integer), `timestamp`, `operation` (WRITE, MERGE, UPDATE, DELETE, RESTORE), and operation parameters.

**What to look for:** Note the version number of the DELETE — you will need it in Tasks 6 and 7.

In [0]:
# TODO: Show the complete transaction history for the customers table
# Wrap in display() so you can read the version numbers

display(spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.customers"))


In [0]:
# -- Validation --
history = spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.customers").collect()
operations = [row["operation"] for row in history]
assert "DELETE" in operations, "Expected DELETE in history"
assert "MERGE" in operations, "Expected MERGE in history"
print(f"Task 5 OK: {len(history)} versions found. Operations: {operations}")

## Task 6: Time Travel - Query Previous Version

Read the table as it was BEFORE the accidental delete (the version with the most rows).

**Guidance — Task 06**

Delta Lake Time Travel allows you to **read** any previous version without modifying the table.

**VERSION AS OF syntax**
```sql
SELECT * FROM catalog.schema.table VERSION AS OF <version_number>
```

**What to do**
1. From the history in Task 5, find the version number **just before** the DELETE (look for MERGE or UPDATE)
2. Assign that number to `version_before_delete`
3. Read that version into `df_recovered` using `spark.sql()` with `VERSION AS OF`

**Hint:** Version numbers are integers starting at 0 (initial WRITE).

In [0]:
# TODO:
# 1. Identify the last version BEFORE the DELETE from the history above
# 2. Assign it to version_before_delete
# 3. Read that version into df_recovered using VERSION AS OF

version_before_delete = 2  # Version after MERGE and UPDATE, before DELETE

df_recovered = spark.sql(f"""
    SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.customers VERSION AS OF {version_before_delete}
""")

print(f"Recovered version has {df_recovered.count()} rows (current has {count_after})")

In [0]:
# -- Validation --
assert df_recovered.count() > count_after, "Recovered version should have more rows than current"
print(f"Task 6 OK: Time Travel successful! Recovered {df_recovered.count()} rows")

## Task 7: RESTORE the Table

Use `RESTORE TABLE` to bring the table back to the version before the accidental delete.

**Guidance — Task 07**

`RESTORE TABLE` permanently rolls the table back to a previous version — it adds a new "RESTORE" entry to the transaction log and makes that version the current state.

**RESTORE syntax**
```sql
RESTORE TABLE catalog.schema.table TO VERSION AS OF <version>
```

Use the same `version_before_delete` from Task 6.

**Hint:** After RESTORE, `DESCRIBE HISTORY` will show a new RESTORE operation at the top.

In [0]:
# TODO: Restore the customers table to the version before the accidental DELETE

spark.sql(f"""
    RESTORE TABLE {CATALOG}.{BRONZE_SCHEMA}.customers TO VERSION AS OF {version_before_delete}
""")

restored_count = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers").count()
print(f"Rows after RESTORE: {restored_count}")

In [0]:
# -- Validation --
assert restored_count == count_before, f"Expected {count_before} rows after restore, got {restored_count}"
print(f"Task 7 OK: Table restored! {restored_count} rows (matches pre-delete count)")

## Task 8: VACUUM and Its Impact on Time Travel

Run `VACUUM` with 0 hours retention, then try to query an old version. You'll see that Time Travel **no longer works** for vacuumed versions — the data files have been physically deleted.

> **Warning:** We use `RETAIN 0 HOURS` for demo purposes only. In production, the default retention is **7 days** — never lower it without understanding the consequences.

In [0]:
# Step 1: Check versions available before VACUUM
history_before = spark.sql(f"DESCRIBE HISTORY {CATALOG}.{BRONZE_SCHEMA}.customers").collect()
print(f"Versions available: {len(history_before)}")
print(f"Version numbers: {[r['version'] for r in history_before]}")

# Step 2: Disable the safety check (LAB ONLY — never do this in production!)
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

**Guidance — Task 08**

`VACUUM` permanently deletes the underlying Parquet data files for versions older than the retention threshold. After VACUUM, those versions can no longer be queried with Time Travel.

**VACUUM syntax**
```sql
VACUUM catalog.schema.table RETAIN <hours> HOURS
```

Use `RETAIN 0 HOURS` to remove all old files immediately (safe only because we disabled the retention check above).

**What to observe:** After running VACUUM, attempting to read `VERSION AS OF 0` will raise a `SparkException` — the data files are gone.

In [0]:
# TODO: Run VACUUM on the customers table with 0 hours retention

spark.sql(f"VACUUM {CATALOG}.{BRONZE_SCHEMA}.customers RETAIN 0 HOURS")

print("VACUUM complete — old data files removed")


In [0]:
# TODO: Try to query VERSION AS OF 0 — this should FAIL after VACUUM
# Use try/except to catch and print the exception (FileNotFoundException / SparkException)

try:
    spark.sql(f"RESTORE TABLE {CATALOG}.{BRONZE_SCHEMA}.customers VERSION AS OF 0").count()
 
except Exception as e:

    print(f"Message: {str(e)[:200]}")


In [0]:
# -- Validation --
# After VACUUM, the latest version should still be accessible
current = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers").count()
assert current > 0, "Current version should still work after VACUUM"
print(f"Task 8 OK: Current table has {current} rows (latest version OK)")
print("Key takeaway: VACUUM removes old files → Time Travel breaks for vacuumed versions")
print(f"  Default retention: 7 days | Production best practice: never set to 0 hours")

## Lab Complete!

You have:
- Used MERGE INTO for upsert (insert + update)
- Performed UPDATE and DELETE on Delta tables
- Inspected history with DESCRIBE HISTORY
- Queried previous versions with Time Travel
- Restored a table with RESTORE TABLE
- Ran VACUUM and observed its impact on Time Travel

> **Exam Tip:** Time Travel uses the Delta transaction log. Data files for old versions are only removed by `VACUUM`. Default retention is **7 days**. After VACUUM, `DESCRIBE HISTORY` still shows metadata, but querying old versions fails because the underlying Parquet files are gone.

> **Next:** LAB 04 - Delta Optimization

## Cleanup (Optional)

In [0]:
# Optional cleanup
# spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{BRONZE_SCHEMA}.customers")
print("LAB 03 complete.")

← [03 — Delta Lake](../demo/03_delta_lake.ipynb) | **[ README](../../../README.md)** | [04 — Delta Optimization →](../../day2/demo/04_delta_optimization.ipynb)